# Pilot analysis - flatness vs. OOD generalization

**Hypothesis.** Distillation steers a smaller student toward *flatter* minima than its teacher, and that flatness (not just capacity) helps explain cases where the student beats the teacher out of distribution. Check robustness to training precision (fp32 vs AMP).

Input: `results/pilot_summary.csv`, produced by
`python -m src.evaluate --aggregate --results-dir results/ --out results/pilot_summary.csv`
(after `src.train`, `src.evaluate --all`, `src.measure_geometry --all`).

Columns of interest: `id_acc`, `ood_acc_mean`, `mce_vs_baseline`,
`adaptive_sharpness` (headline geometry metric), `hessian_trace`,
`hessian_top_eigenvalue`, plus `mode`, `precision`, `width_mult`, `seed`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV = Path('../results/pilot_summary.csv')
assert CSV.exists(), f'{CSV} not found - run the pilot + aggregate first.'
df = pd.read_csv(CSV)
df['label'] = np.where(df['mode'] == 'teacher', 'teacher',
                       'student w' + df['width_mult'].astype(str))
df = df.sort_values(['precision', 'mode', 'width_mult', 'seed']).reset_index(drop=True)
df[['run_name','mode','precision','width_mult','seed','id_acc','ood_acc_mean',
    'mce_vs_baseline','adaptive_sharpness','hessian_trace','hessian_top_eigenvalue']]

## 1. Mean ± std over seeds

In [ ]:
metrics = ['id_acc', 'ood_acc_mean', 'mce_vs_baseline',
           'adaptive_sharpness', 'hessian_trace', 'hessian_top_eigenvalue']
agg = (df.groupby(['precision', 'label'])[metrics]
         .agg(['mean', 'std']).round(4))
agg

## 2. Does the student beat its teacher on OOD? Is it flatter?

For each (precision, seed) we compare every student against the teacher of the
same precision and seed.

In [ ]:
teachers = (df[df['mode'] == 'teacher']
            .set_index(['precision', 'seed'])[metrics])
rows = []
for _, s in df[df['mode'] == 'student'].iterrows():
    key = (s['precision'], s['seed'])
    if key not in teachers.index:
        continue
    t = teachers.loc[key]
    rows.append({
        'precision': s['precision'], 'seed': s['seed'], 'student': s['label'],
        'd_id_acc': s['id_acc'] - t['id_acc'],
        'd_ood_acc': s['ood_acc_mean'] - t['ood_acc_mean'],
        'beats_teacher_ood': s['ood_acc_mean'] > t['ood_acc_mean'],
        'd_sharpness': s['adaptive_sharpness'] - t['adaptive_sharpness'],
        'flatter_than_teacher': s['adaptive_sharpness'] < t['adaptive_sharpness'],
        'd_hessian_trace': s['hessian_trace'] - t['hessian_trace'],
    })
cmp = pd.DataFrame(rows)
cmp

In [ ]:
# Contingency: OOD win vs. flatter-than-teacher
pd.crosstab(cmp['beats_teacher_ood'], cmp['flatter_than_teacher'],
            rownames=['beats teacher OOD'], colnames=['flatter than teacher'])

## 3. Core plot - sharpness vs. OOD accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, prec in zip(axes, ['fp32', 'amp']):
    sub = df[df['precision'] == prec]
    if sub.empty:
        ax.set_title(f'{prec} (no runs)'); continue
    for lab, g in sub.groupby('label'):
        ax.scatter(g['adaptive_sharpness'], g['ood_acc_mean'], s=60, label=lab)
    ax.set_xlabel('adaptive sharpness  (ρ=0.05, lower = flatter)')
    ax.set_title(prec)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('CIFAR-10-C mean accuracy (OOD)')
axes[0].legend()
fig.suptitle('Flatter minima vs. OOD generalization')
fig.tight_layout()

In [ ]:
# Same, against Hessian trace (secondary metric)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for lab, g in df.groupby('label'):
    ax.scatter(g['hessian_trace'], g['ood_acc_mean'], s=60, label=lab)
ax.set_xlabel('Hessian trace (lower = flatter)')
ax.set_ylabel('CIFAR-10-C mean accuracy (OOD)')
ax.grid(alpha=0.3); ax.legend()
ax.set_title('Hessian trace vs. OOD accuracy')
fig.tight_layout()

## 4. Precision robustness - fp32 vs. AMP

In [ ]:
if df['precision'].nunique() < 2:
    print('Only one precision present - train the AMP configs to compare.')
else:
    piv = (df.groupby(['label', 'precision'])[['ood_acc_mean', 'adaptive_sharpness']]
             .mean().unstack('precision'))
    display(piv.round(4))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axes, ['ood_acc_mean', 'adaptive_sharpness']):
        piv[m].plot(kind='bar', ax=ax); ax.set_title(m); ax.grid(alpha=0.3)
        ax.set_xlabel(''); ax.tick_params(axis='x', rotation=20)
    fig.suptitle('fp32 vs AMP'); fig.tight_layout()

## 5. Read-out

The pilot supports the hypothesis to the extent that:

1. Students that beat the teacher on CIFAR-10-C mostly fall in the
   *flatter-than-teacher* cell of the section-2 contingency table.
2. The section-3 scatter shows OOD accuracy decreasing with sharpness across
   models (not just tracking capacity/width).
3. The pattern holds in **both** fp32 and AMP (section 4) - i.e. it is not an
   artifact of numerical precision.

This is a pilot (n=3 seeds): report effect directions and per-seed consistency,
not significance.